# 🤖 Notebook 3 — Pipeline RAG Completo

Implementa as Etapas 9 e 10 do pipeline:

**Retrieval híbrido + reranking + expansão:**
1. **Dense:** `multilingual-e5-large-instruct` (Qdrant)
2. **Sparse:** BM25
3. **Fusão:** Reciprocal Rank Fusion (RRF)
4. **Reranking:** Cross-encoder `mmarco-mMiniLMv2-L12-H384-v1` (PT-BR)
5. **Expansão por ato_id:** se top result é voto, busca texto_integral do mesmo ato
6. **Boost por tipo:** texto_integral > nota_tecnica > decisao > voto

**LLMs disponíveis (selecionáveis):**
- 🦫 Sabiá-3 (Maritaca AI) — gratuito, melhor PT-BR
- 🦙 Llama 3.3 70B (Groq) — gratuito, alta velocidade

**Prompt jurídico estruturado** com citação obrigatória, recusa informada, diferenciação tipo de fonte.

In [ ]:
# ============================================================
# CÉLULA 1 — Instalação
# ============================================================
%%capture
!pip install sentence-transformers qdrant-client rank-bm25 groq maritalk tqdm pyarrow

In [ ]:
# ============================================================
# CÉLULA 2 — ⚙️ CONFIGURAÇÕES (edite aqui)
# ============================================================

# 🔑 Chaves de API (obtenha gratuitamente nos sites)
# Maritaca:  https://plataforma.maritaca.ai/chaves-api
# Groq:      https://console.groq.com/keys
MARITACA_API_KEY = '100000000000000000000$XXXXXXXXXXXXXXXXXXXXXXXXXXXX'
GROQ_API_KEY     = 'gsk_XXXXXXXXXXXXXXXXXXXXXXXXXXXX'

# Qual LLM usar por padrão?
# Opções: 'maritaca' | 'groq'
LLM_DEFAULT = 'maritaca'

# Modelos
EMBEDDING_MODEL = 'intfloat/multilingual-e5-large-instruct'
RERANKER_MODEL  = 'cross-encoder/mmarco-mMiniLMv2-L12-H384-v1'  # multilingual com PT-BR
MARITACA_MODEL  = 'sabia-3'
GROQ_MODEL      = 'llama-3.3-70b-versatile'

# Parâmetros de retrieval
DENSE_TOP_K     = 30   # candidatos do dense search
SPARSE_TOP_K    = 30   # candidatos do BM25
AFTER_FUSION_K  = 25   # após RRF, antes do reranking
AFTER_RERANK_K  = 6    # após cross-encoder
FINAL_K         = 5    # chunks enviados ao LLM (após expansão)
ALPHA_RRF       = 0.6  # peso semântico vs sparse na RRF

# Boost por tipo (multiplica o score do reranker)
TIPO_BOOST = {
    'texto_integral': 1.20,  # normativo definitivo
    'nota_tecnica':   1.05,  # análise técnica
    'anexo':          1.05,  # tabelas, dados
    'decisao':        1.00,  # neutro
    'voto':           0.90,  # argumentativo (não definitivo)
    'outro':          0.85,
}

DRIVE_DIR   = '/content/drive/MyDrive/aneel_rag'
QDRANT_PATH = '/content/qdrant_storage'

print('✅ Configurações definidas')
print(f'   LLM default: {LLM_DEFAULT.upper()}')
print(f'   Dense+Sparse → RRF → Rerank → Expand → Top-{FINAL_K}')

In [ ]:
# ============================================================
# CÉLULA 3 — Imports
# ============================================================
import os
import re
import gc
import json
import time
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny
from rank_bm25 import BM25Okapi

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device: {device.upper()}')

In [ ]:
# ============================================================
# CÉLULA 4 — Restaurar Qdrant + chunks do Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Restaura Qdrant
if not os.path.exists(QDRANT_PATH):
    print('📦 Descomprimindo Qdrant do Drive...')
    os.system(f'cp {DRIVE_DIR}/qdrant_storage.tar.gz /content/')
    os.system('tar -xzf /content/qdrant_storage.tar.gz -C /content/')
    print('✅ Qdrant restaurado')
else:
    print('♻️  Qdrant já existe localmente')

client = QdrantClient(path=QDRANT_PATH)
info = client.get_collection('aneel_legislacao')
print(f'✅ Coleção: {info.points_count:,} pontos')

# Carrega chunks (para BM25)
print('\n📥 Carregando chunks para BM25...')
df_chunks = pd.read_parquet(f'{DRIVE_DIR}/chunks_hierarquicos.parquet')
print(f'✅ {len(df_chunks):,} chunks')

In [ ]:
# ============================================================
# CÉLULA 5 — Constrói BM25 (sparse retrieval)
# ============================================================
# Tokenização preserva acentos e termos jurídicos importantes

STOPWORDS_PT_MIN = {
    'a', 'o', 'e', 'de', 'do', 'da', 'em', 'no', 'na', 'um', 'uma',
    'os', 'as', 'dos', 'das', 'que', 'para', 'com', 'por', 'ao', 'à',
    'se', 'sua', 'seu', 'suas', 'seus', 'mas', 'ou', 'ser', 'foi',
}

def tokenize_pt(text: str) -> list:
    text = (text or '').lower()
    text = re.sub(r"[^\w\sáéíóúâêîôûãõàèìòùçñ]", ' ', text)
    tokens = [t for t in text.split() if len(t) > 2 and t not in STOPWORDS_PT_MIN]
    return tokens

print('🔍 Tokenizando corpus...')
corpus_tokens = [tokenize_pt(t) for t in tqdm(df_chunks['texto'].tolist(), desc='Tokenize')]

print('🏗️  Construindo BM25...')
bm25 = BM25Okapi(corpus_tokens)
print(f'✅ BM25 pronto para {len(corpus_tokens):,} chunks')

In [ ]:
# ============================================================
# CÉLULA 6 — Carrega modelos (embedder + reranker)
# ============================================================

print(f'📥 Embedder: {EMBEDDING_MODEL}')
embed_model = SentenceTransformer(EMBEDDING_MODEL, device=device)
embed_model.max_seq_length = 384

print(f'📥 Reranker: {RERANKER_MODEL}')
reranker = CrossEncoder(RERANKER_MODEL, max_length=512, device=device)

print('✅ Modelos carregados')

In [ ]:
# ============================================================
# CÉLULA 7 — Funções de retrieval
# ============================================================

TASK_INSTRUCTION = (
    'Given a question about Brazilian electric sector regulations '
    '(ANEEL legislation), retrieve the most relevant legal passages '
    'that directly answer the question.'
)

def encode_query(query: str) -> list:
    """Formata e codifica a query no padrão do e5-instruct."""
    formatted = f'Instruct: {TASK_INSTRUCTION}\nQuery: {query}'
    return embed_model.encode(formatted, normalize_embeddings=True).tolist()


def dense_search(query: str, k: int, tipo_filter: list = None):
    """Busca densa no Qdrant, opcionalmente filtrada por tipo."""
    q_emb = encode_query(query)

    qdrant_filter = None
    if tipo_filter:
        qdrant_filter = Filter(must=[
            FieldCondition(key='tipo_documento', match=MatchAny(any=tipo_filter))
        ])

    results = client.query_points(
        collection_name='aneel_legislacao',
        query=q_emb,
        limit=k,
        query_filter=qdrant_filter,
        with_payload=True,
    ).points
    return [(r.id, r.score, r.payload) for r in results]


def sparse_search(query: str, k: int):
    """Busca BM25."""
    tokens = tokenize_pt(query)
    if not tokens:
        return []
    scores = bm25.get_scores(tokens)
    top_idx = np.argpartition(scores, -k)[-k:]
    top_idx = top_idx[np.argsort(scores[top_idx])[::-1]]
    out = []
    for idx in top_idx:
        if scores[idx] <= 0:
            continue
        row = df_chunks.iloc[int(idx)]
        payload = {
            'texto':            row['texto'],
            'texto_pai':        row['texto_pai'],
            'ato_id':           row['ato_id'],
            'tipo_documento':   row['tipo_documento'],
            'titulo':           row['titulo'],
            'ementa':           row['ementa'],
            'situacao':         row['situacao'],
            'publicacao':       row['publicacao'],
            'autor':            row['autor'],
            'ano':              int(row['ano']),
            'contexto_juridico': row['contexto_juridico'],
        }
        out.append((int(idx), float(scores[idx]), payload))
    return out


def rrf_fuse(dense_res, sparse_res, alpha=ALPHA_RRF, k=60):
    """
    Reciprocal Rank Fusion: combina rankings sem normalizar scores.
    Robusto a escalas diferentes (cosine vs BM25).
    """
    scores = {}
    payloads = {}

    for rank, (idx, _, payload) in enumerate(dense_res):
        scores[idx] = scores.get(idx, 0) + alpha / (k + rank + 1)
        payloads[idx] = payload

    for rank, (idx, _, payload) in enumerate(sparse_res):
        scores[idx] = scores.get(idx, 0) + (1 - alpha) / (k + rank + 1)
        payloads[idx] = payloads.get(idx, payload)

    fused = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [(idx, score, payloads[idx]) for idx, score in fused]


def cross_encode_rerank(query: str, candidates: list, top_k: int):
    """
    Reranqueia candidatos com cross-encoder, aplicando boost por tipo_documento.
    Resolve o problema do PDF: voto + nota_tecnica dominando 53% do índice.
    """
    if not candidates:
        return []

    pairs = [(query, c[2]['texto']) for c in candidates]
    rerank_scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    boosted = []
    for (idx, _, payload), s in zip(candidates, rerank_scores):
        boost = TIPO_BOOST.get(payload['tipo_documento'], 1.0)
        boosted.append((idx, float(s) * boost, payload, float(s)))

    boosted.sort(key=lambda x: x[1], reverse=True)
    return boosted[:top_k]


def expand_by_ato_id(reranked: list, max_extra: int = 2):
    """
    Expansão por ato_id: se um voto/nota_tecnica está no topo,
    busca o texto_integral do mesmo ato (que define o conteúdo normativo).
    """
    if not reranked:
        return reranked

    extras = []
    seen_ids = {r[0] for r in reranked}
    seen_atos = set()

    for entry in reranked[:3]:  # só expande os top 3
        ato_id = entry[2]['ato_id']
        tipo = entry[2]['tipo_documento']

        if tipo in ('voto', 'nota_tecnica') and ato_id and ato_id not in seen_atos:
            seen_atos.add(ato_id)
            mask = (df_chunks['ato_id'] == ato_id) & (df_chunks['tipo_documento'] == 'texto_integral')
            ti_chunks = df_chunks[mask]
            if len(ti_chunks) > 0:
                # Pega o primeiro chunk do texto_integral (geralmente o mais relevante)
                row = ti_chunks.iloc[0]
                extra_idx = int(row.name)
                if extra_idx not in seen_ids and len(extras) < max_extra:
                    extras.append((extra_idx, 0.0, {
                        'texto':            row['texto'],
                        'texto_pai':        row['texto_pai'],
                        'ato_id':           row['ato_id'],
                        'tipo_documento':   row['tipo_documento'],
                        'titulo':           row['titulo'],
                        'ementa':           row['ementa'],
                        'situacao':         row['situacao'],
                        'publicacao':       row['publicacao'],
                        'autor':            row['autor'],
                        'ano':              int(row['ano']),
                        'contexto_juridico': row['contexto_juridico'],
                    }, 0.0))
                    seen_ids.add(extra_idx)

    return reranked + extras


def deduplicate_by_doc(results: list, max_per_doc: int = 2):
    """Limita chunks de um mesmo documento (evita repetição)."""
    seen = {}
    out = []
    for r in results:
        key = (r[2]['ato_id'], r[2]['arquivo_origem']) if 'arquivo_origem' in r[2] else r[2]['ato_id']
        if seen.get(key, 0) < max_per_doc:
            seen[key] = seen.get(key, 0) + 1
            out.append(r)
    return out


def hybrid_retrieve(
    query: str,
    final_k: int = FINAL_K,
    tipo_filter: list = None,
    use_expansion: bool = True,
    use_reranking: bool = True,
):
    """Pipeline de retrieval completo: dense + sparse + RRF + rerank + expand."""
    dense_res  = dense_search(query, DENSE_TOP_K, tipo_filter)
    sparse_res = sparse_search(query, SPARSE_TOP_K)
    fused      = rrf_fuse(dense_res, sparse_res)[:AFTER_FUSION_K]

    if use_reranking:
        reranked = cross_encode_rerank(query, fused, AFTER_RERANK_K)
    else:
        reranked = [(idx, score, payload, score) for idx, score, payload in fused[:AFTER_RERANK_K]]

    if use_expansion:
        reranked = expand_by_ato_id(reranked, max_extra=2)

    reranked = deduplicate_by_doc(reranked, max_per_doc=2)
    return reranked[:final_k]


print('✅ Funções de retrieval definidas')

In [ ]:
# ============================================================
# CÉLULA 8 — Teste do pipeline de retrieval
# ============================================================
query_teste = 'Quais são as penalidades para distribuidoras que descumprem prazos de atendimento?'

results = hybrid_retrieve(query_teste)

print(f'🔍 Query: "{query_teste}"\n')
for i, r in enumerate(results, 1):
    idx, score, payload = r[:3]
    print(f"{i}. [{payload['tipo_documento']:14}] {payload['ato_id']:25} | {payload['publicacao']}")
    print(f"   {payload['titulo'][:80]}")
    print(f"   Score: {score:.4f}")
    print(f"   {payload['texto'][:200]}...\n")

In [ ]:
# ============================================================
# CÉLULA 9 — Clientes LLM (Maritaca + Groq)
# ============================================================
from groq import Groq

# Cliente Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY and not GROQ_API_KEY.startswith('gsk_X') else None

# Cliente Maritaca
maritaca_client = None
if MARITACA_API_KEY and not MARITACA_API_KEY.startswith('100000000'):
    try:
        import maritalk
        maritaca_client = maritalk.MariTalk(key=MARITACA_API_KEY, model=MARITACA_MODEL)
    except Exception as e:
        print(f'⚠️  Maritaca falhou ao carregar: {e}')

print(f'🦫 Maritaca: {"✅ pronto" if maritaca_client else "❌ não configurado"}')
print(f'🦙 Groq:     {"✅ pronto" if groq_client else "❌ não configurado"}')

# Teste rápido
if groq_client:
    try:
        r = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{'role': 'user', 'content': 'Diga apenas: OK'}],
            max_tokens=5,
        )
        print(f'   Groq teste: {r.choices[0].message.content.strip()}')
    except Exception as e:
        print(f'   ❌ Groq teste falhou: {e}')

if maritaca_client:
    try:
        r = maritaca_client.generate('Diga apenas: OK', max_tokens=5)
        ans = r['answer'] if isinstance(r, dict) else str(r)
        print(f'   Maritaca teste: {ans.strip()}')
    except Exception as e:
        print(f'   ❌ Maritaca teste falhou: {e}')

In [ ]:
# ============================================================
# CÉLULA 10 — Prompt jurídico estruturado
# ============================================================

SYSTEM_PROMPT = """Você é um especialista em regulação do setor elétrico brasileiro, com profundo conhecimento das resoluções, normas e procedimentos da ANEEL (Agência Nacional de Energia Elétrica).

Sua missão é responder perguntas técnicas e regulatórias com base EXCLUSIVAMENTE nos documentos fornecidos no contexto.

═══ REGRAS OBRIGATÓRIAS ═══

1. **Use APENAS as informações nos documentos do contexto.** Não invente, não complete, não use conhecimento prévio.

2. **Recusa informada:** Se a informação não estiver nos documentos, responda exatamente:
   "Não encontrei informação suficiente nos documentos disponíveis para responder esta pergunta com precisão."
   Em seguida, indique que tipo de documento poderia conter a resposta.

3. **Citação obrigatória:** A cada afirmação, cite a fonte no formato:
   `(Tipo: XXX | Ato: YYY | Publicação: AAAA-MM-DD)`
   Mencione artigos, parágrafos ou incisos quando aplicável.

4. **Hierarquia das fontes:** Diferencie tipo de fonte:
   - **texto_integral** = norma vigente (CONTEÚDO DEFINITIVO)
   - **anexo** = tabelas, dados regulatórios oficiais
   - **nota_tecnica** = análise técnica fundamentando a decisão
   - **decisao** = decisão administrativa
   - **voto** = argumento do relator (NÃO é norma — apenas posição individual)

   Se a resposta vier APENAS de voto, deixe explícito que é a posição argumentativa do relator, não o texto normativo.

5. **Situação do ato:** Se o documento estiver REVOGADO, sinalize claramente. A resposta deve esclarecer se a norma ainda está em vigor.

6. **Contradições:** Se documentos diferentes apresentarem informações contraditórias, aponte explicitamente, com as datas e fontes.

7. **Tom:** Técnico, objetivo, juridicamente preciso. Sem especulações, sem floreios."""


def format_context(retrieved_chunks):
    """Formata os chunks recuperados em contexto estruturado para o LLM."""
    blocks = []
    for i, entry in enumerate(retrieved_chunks, 1):
        payload = entry[2]
        # Usa texto_pai (mais contexto) em vez do texto_filho
        texto = payload.get('texto_pai') or payload['texto']

        block = (
            f"━━━ DOCUMENTO {i} ━━━\n"
            f"Tipo: {payload['tipo_documento'].upper()}\n"
            f"Ato: {payload['ato_id']}\n"
            f"Título: {payload['titulo']}\n"
            f"Publicação: {payload['publicacao']}\n"
            f"Situação: {payload['situacao']}\n"
        )
        if payload.get('ementa'):
            block += f"Ementa: {payload['ementa']}\n"
        if payload.get('contexto_juridico'):
            block += f"Localização: {payload['contexto_juridico']}\n"
        block += f"\nTRECHO:\n{texto}\n"
        blocks.append(block)

    return '\n'.join(blocks)


print('✅ Prompt e formatação definidos')

In [ ]:
# ============================================================
# CÉLULA 11 — Geração de resposta
# ============================================================

def generate_with_groq(messages, max_tokens=1024, temperature=0.1):
    """Geração via Groq Llama."""
    if not groq_client:
        raise RuntimeError('Groq não configurado')
    resp = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=messages,
        max_tokens=max_tokens,
        temperature=temperature,
    )
    return resp.choices[0].message.content


def generate_with_maritaca(messages, max_tokens=1024, temperature=0.1):
    """Geração via Maritaca Sabiá-3."""
    if not maritaca_client:
        raise RuntimeError('Maritaca não configurada')
    # Maritaca aceita lista de chat messages
    resp = maritaca_client.generate(
        messages,
        max_tokens=max_tokens,
        temperature=temperature,
        do_sample=temperature > 0,
    )
    return resp['answer'] if isinstance(resp, dict) else str(resp)


def ask(
    query: str,
    llm: str = LLM_DEFAULT,
    final_k: int = FINAL_K,
    tipo_filter: list = None,
    verbose: bool = True,
    return_chunks: bool = False,
):
    """
    Pipeline RAG completo: hybrid retrieval + reranking + LLM.

    Args:
        query: Pergunta
        llm: 'maritaca' ou 'groq'
        final_k: Quantos chunks vão ao LLM
        tipo_filter: Lista de tipos para filtrar (ex: ['texto_integral'])
        verbose: Imprime detalhes
    """
    if verbose:
        print(f"\n{'='*70}")
        print(f"❓ {query}")
        print(f"{'='*70}\n")

    t0 = time.time()
    retrieved = hybrid_retrieve(query, final_k=final_k, tipo_filter=tipo_filter)
    t_retrieval = time.time() - t0

    if verbose:
        print(f'📚 {len(retrieved)} chunks (retrieval em {t_retrieval:.2f}s):')
        for i, r in enumerate(retrieved, 1):
            p = r[2]
            print(f"  {i}. [{p['tipo_documento']:14}] {p['ato_id'][:25]:25} | {p['publicacao']} | score={r[1]:.3f}")

    if not retrieved:
        ans = 'Não encontrei documentos relevantes para esta pergunta no acervo.'
        return (ans, []) if return_chunks else ans

    context = format_context(retrieved)
    user_msg = f"DOCUMENTOS RECUPERADOS:\n\n{context}\n\n═══════════════════\nPERGUNTA: {query}\n\nResponda com base exclusivamente nos documentos acima, citando as fontes."

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]

    t0 = time.time()
    if llm == 'maritaca':
        answer = generate_with_maritaca(messages)
    elif llm == 'groq':
        answer = generate_with_groq(messages)
    else:
        raise ValueError(f'LLM desconhecido: {llm}')
    t_llm = time.time() - t0

    if verbose:
        print(f'\n💡 RESPOSTA ({llm}, {t_llm:.1f}s):')
        print(answer)
        print(f"\n{'='*70}")

    return (answer, retrieved) if return_chunks else answer


print('✅ Pipeline RAG pronto. Use: ask("sua pergunta")')

In [ ]:
# ============================================================
# CÉLULA 12 — Testes
# ============================================================

perguntas_teste = [
    'Quais são os critérios para revisão tarifária periódica das distribuidoras?',
    'O que é o Mecanismo de Realocação de Energia (MRE)?',
    'Quais penalidades a ANEEL aplica em caso de descumprimento de obrigações?',
]

for q in perguntas_teste:
    try:
        ask(q)
    except Exception as e:
        print(f'⚠️  Erro: {e}\n')

In [ ]:
# ============================================================
# CÉLULA 13 — Comparar Maritaca vs Groq lado a lado
# ============================================================
# Útil para escolher qual LLM usar no benchmark final

def compare_llms(query: str):
    print(f"\n{'='*70}\n❓ {query}\n{'='*70}")
    retrieved = hybrid_retrieve(query)
    print(f'📚 {len(retrieved)} chunks recuperados\n')

    context = format_context(retrieved)
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': f'DOCUMENTOS:\n{context}\n\nPERGUNTA: {query}'},
    ]

    if maritaca_client:
        try:
            print('━━━━━ 🦫 SABIÁ-3 ━━━━━')
            print(generate_with_maritaca(messages))
        except Exception as e:
            print(f'❌ {e}')

    if groq_client:
        try:
            print('\n━━━━━ 🦙 LLAMA 3.3 70B ━━━━━')
            print(generate_with_groq(messages))
        except Exception as e:
            print(f'❌ {e}')

# Exemplo:
# compare_llms('A REN 414/2010 ainda está em vigor?')

In [ ]:
# ============================================================
# CÉLULA 14 — Modo interativo
# ============================================================
print('💬 Modo interativo. Digite "sair" para encerrar.\n')
while True:
    q = input('🔍 Pergunta: ').strip()
    if not q:
        continue
    if q.lower() in ('sair', 'exit', 'quit', 'q'):
        print('👋 Até logo!')
        break
    try:
        ask(q)
    except Exception as e:
        print(f'⚠️  Erro: {e}')